# Profile proximity vs physical proximity (Danenberg 2022 IMC)

**Question (for the CyTOF / C2S project).** CyTOF loses spatial coordinates, so "neighbours" there are cells with similar marker profiles.
In intact tissue, are a cell's profile neighbours also its physical neighbours? Do they share its physical surroundings?
And does that hold beyond cell type?

**Data.** Output of the `danenberg-cells-prep` kernel:
- published single-cell table, spillover-corrected;
- main invasive-tumour images only (`is_tumour == 1`), with hot-aggregate cells removed;
- images with at least 200 cells;
- 1 px = 1 µm.

**Encodings** (copied from the CyTOF code):
- *intensity*: arcsinh(x / cofactor), then per-image robust z `(x−median)/(1.4826·MAD+1e-6)`, clipped to ±5 (`bc_knn_enrichment.py`);
- *rank*: each marker's position within the cell (0 = highest), computed from the clipped z, ties broken by column order (`rankvec_control.py` @ 9dad454);
- *rank_avg*: same, with average ranks for ties (sensitivity check).

Neighbours use Euclidean kNN. Exact distance ties are broken **randomly**: `ObjectNumber` order follows position in the image, so index-order tie-breaking would fake proximity.

**Profile spaces:**
- `cytof6`: the 6 markers shared with the BC CyTOF panel (panCK, CK5, CK8/18, SMA, ER, Ki-67);
- `full36`: all 36 protein channels;
- `randN_*`: random N-marker subsets of `full36`, with N = 6 (20 sets), 12, 18, 24 and 31 (8 sets each).

**Metrics** (per image; k = profile neighbours):
- **A1:** overlap between profile kNN and physical kNN, |P∩S|/k.
- **A2:** mean over cells of log2( median physical distance to profile neighbours / median distance to all other cells ). Below 0 means closer than a random cell.
- **A3:** mean profile distance per physical-distance band, relative to the image mean.
- **B1:** R² of predicting a cell's *physical context* from the mean context of its profile neighbours. Context = phenotype composition of its 10 physically nearest cells. Reported at two resolutions: fine (32 phenotypes) and broad (epithelial/immune/stromal).
- **B2:** fraction of profile neighbours in the same cellular neighbourhood (CN = k-means, 10 clusters, on fine context).

**Nulls.** Replace each profile neighbour by
- (N1) a uniformly random cell of the image, or
- (N2) a random cell of **the same phenotype as that neighbour**. This keeps the cell-type composition of the neighbour set.

`obs − N2` is the signal **beyond cell type**.

**Spillover guard.** Contact mode 1 removes touching cells (published contact graph) from the profile-neighbour candidates. The primary setting is k = 30, contacts excluded.

**Cohort pool (transfer).** Profile neighbours for cells of half the patients are drawn from a reference pool of the other half, with B1/B2 and the same nulls. This is the analogue of assigning context to CyTOF cells from an IMC reference.

**Stats.** Per-image values are averaged per patient, then bootstrapped over patients (B = 1000, 95% percentile CI).

## Decision rule (fixed before the run; primary = k 30, contacts excluded, within image)
1. **Profile neighbours are physically closer than chance** if the CI of A2 `obs − N1` is entirely < 0.
2. **…beyond cell type** if the CI of A2 `obs − N2` is entirely < 0.
3. **Profile neighbours carry physical context beyond cell type** if the CI of B1-fine `obs − N2` is entirely > 0.
4. **CyTOF verdict:**
   - If (3) fails for *both* encodings in `cytof6` **and** `full36`: profile neighbours carry no physical context beyond cell type, so drop the CyTOF "virtual space" transfer (future_plan §13 Step 3).
   - If (3) holds only in `full36`: the result is panel-limited. The 31-marker CyTOF panel cannot be tested here.
5. **Encoding answer for the C2S paper:** the CI of the per-patient (rank − intensity) difference in B1-fine `obs − N2` (and in A2).

## Added in version 3 (fixed before the run)
- **Panel-size curve.** Random marker sets of 6, 12, 18, 24 and 31 (31 = the CyTOF panel size), plus all 36.
- **Cell-type purity of profile neighbours:** T1 (same fine phenotype) and T1b (same broad class). This is the grouping C2S relies on.
- **Bug fix.** The type-matched null no longer picks the cell itself.
- **Dropped `rank_avg`.** It matched `rank` to 3 decimals in version 2.
- **"Ranking does no harm" at a panel size** means, for context R² (B1f obs) and type purity (T1 obs), that the CI of the per-patient
  (rank − intensity) difference lies above −0.01. A loss of at most 0.01 counts as equivalent.

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.stats import rankdata
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260927
OUT = "/kaggle/working"
COFACTOR = 1.0            # IMC mean counts per pixel (prep report: medians ~0.1-20)
KS = (10, 30)
N_NULL = 20
N_NULL_SUBSET = 5
SUBSET_SIZES = {6: 20, 12: 8, 18: 8, 24: 8, 31: 8}   # panel size -> number of random marker sets
BANDS = np.array([0, 15, 30, 60, 120, 240, np.inf])
MIN_CELLS, MAX_CELLS = 200, 6000
N_BOOT = 1000
CTX_K, N_CN = 10, 10
POOL_Q, POOL_R, POOL_NULL = 50_000, 200_000, 10
CYTOF6 = ["pan_cytokeratin", "cytokeratin_5", "cytokeratin_8_18", "sma",
          "estrogen_receptor_alpha", "ki_67"]
N_JOBS = os.cpu_count()

def find_input(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"missing input {name}"
    return hits[0]

## Analysis functions

In [ ]:
def rank_vec(Zs, ties="order"):
    """0 = highest. 'order': ties by column order (stable sort, as in the CyTOF code); 'average': average ranks."""
    if ties == "average":
        return (rankdata(-Zs, axis=1, method="average") - 1).astype(np.float32)
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def per_image(i, img_seed, xy_i, ph_i, br_i, ctxf_i, ctxb_i, cn_i, Z_i, pairs, spaces, spec):
    rng = np.random.default_rng(img_seed)
    n = len(xy_i)
    if n > MAX_CELLS:
        keep = np.sort(rng.choice(n, MAX_CELLS, replace=False))
        remap = -np.ones(n, np.int64); remap[keep] = np.arange(MAX_CELLS)
        xy_i, ph_i, br_i, ctxf_i, ctxb_i, cn_i, Z_i = (x[keep] for x in (xy_i, ph_i, br_i, ctxf_i, ctxb_i, cn_i, Z_i))
        pa, pb = remap[pairs[0]], remap[pairs[1]]
        m = (pa >= 0) & (pb >= 0)
        pairs = (pa[m], pb[m])
        n = MAX_CELLS
    ar = np.arange(n)[:, None]
    Dp = squareform(pdist(xy_i)).astype(np.float32)
    np.fill_diagonal(Dp, np.inf)
    med_all = np.median(np.sort(Dp, axis=1)[:, :-1], axis=1)       # excludes self (inf sorts last)
    S = {k: np.argpartition(Dp, k, axis=1)[:, :k] for k in KS}
    Smask = {}
    for k in KS:
        M = np.zeros((n, n), bool); M[ar, S[k]] = True; Smask[k] = M
    C = np.zeros((n, n), bool); C[pairs[0], pairs[1]] = True; C[pairs[1], pairs[0]] = True
    by_ph = {p: np.flatnonzero(ph_i == p) for p in np.unique(ph_i)}
    iu = np.triu_indices(n, 1)
    band = np.digitize(Dp[iu], BANDS) - 1
    sst_f = ((ctxf_i - ctxf_i.mean(0)) ** 2).sum()
    sst_b = ((ctxb_i - ctxb_i.mean(0)) ** 2).sum()

    def metrics(P, k):
        d = Dp[ar, P]
        return dict(
            A1=Smask[k][ar, P].mean(),
            A2=np.log2(np.median(d, 1) / med_all).mean(),
            B1f=1 - ((ctxf_i - ctxf_i[P].mean(1)) ** 2).sum() / sst_f,
            B1b=1 - ((ctxb_i - ctxb_i[P].mean(1)) ** 2).sum() / sst_b,
            B2=(cn_i[P] == cn_i[:, None]).mean(),
            T1=(ph_i[P] == ph_i[:, None]).mean(),
            T1b=(br_i[P] == br_i[:, None]).mean())

    def null_uniform(k):
        P = rng.integers(0, n - 1, (n, k))
        return P + (P >= np.arange(n)[:, None])

    def null_type(P):
        Q = np.empty_like(P); phP = ph_i[P]
        for p, idx in by_ph.items():
            m = phP == p
            Q[m] = idx[rng.integers(0, len(idx), m.sum())]
        self_ = np.arange(n)[:, None]
        for _ in range(20):                      # never use the cell itself as its own null neighbour
            bad = Q == self_
            if not bad.any():
                break
            for p in np.unique(phP[bad]):
                m = bad & (phP == p)
                Q[m] = by_ph[p][rng.integers(0, len(by_ph[p]), m.sum())]
        bad = Q == self_
        Q[bad] = P[bad]                          # fallback: the original neighbour (same type, not the cell)
        return Q

    rows, a3 = [], []
    for sname, cols in spaces.items():
        sp = spec[sname]
        Zs = Z_i[:, cols]
        for enc in sp["encs"]:
            V = Zs if enc == "intensity" else rank_vec(Zs, "average" if enc == "rank_avg" else "order")
            Dv = cdist(V, V).astype(np.float32)
            if sp["a3"]:
                dv = Dv[iu]
                tot = dv.mean()
                for bi in range(len(BANDS) - 1):
                    m = band == bi
                    a3.append(dict(img=i, space=sname, enc=enc, band=bi, n=int(m.sum()),
                                   rel=float(dv[m].mean() / tot) if m.any() else np.nan))
            Dv += rng.uniform(0, 1e-5, Dv.shape).astype(np.float32)   # random tie-breaking
            np.fill_diagonal(Dv, np.inf)
            for mode in sp["modes"]:
                Dm = np.where(C, np.inf, Dv) if mode == 1 else Dv
                for k in sp["ks"]:
                    P = np.argpartition(Dm, k, axis=1)[:, :k]
                    obs = metrics(P, k)
                    n1 = [metrics(null_uniform(k), k) for _ in range(sp["n_null"])]
                    n2 = [metrics(null_type(P), k) for _ in range(sp["n_null"])]
                    for met in obs:
                        rows.append(dict(img=i, space=sname, enc=enc, contact_excl=mode, k=k, metric=met,
                                         obs=obs[met], n1=np.mean([d[met] for d in n1]),
                                         n2=np.mean([d[met] for d in n2])))
    return rows, a3

def boot_mean(per_patient, n_boot=N_BOOT, seed=SEED):
    v = np.asarray(per_patient, float)
    v = v[~np.isnan(v)]
    r = np.random.default_rng(seed)
    bs = v[r.integers(0, len(v), (n_boot, len(v)))].mean(1)
    return v.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)

## Sanity check on synthetic images (must pass before the real run)
- (a) Profiles are a smooth function of position and phenotypes are random, so A2 `obs − N1` and `obs − N2` must be clearly < 0.
- (b) Same data with profiles shuffled across cells, so both excesses must be ≈ 0.

In [ ]:
def synthetic(shuffle, seed=0):
    r = np.random.default_rng(seed)
    n = 1200
    pos = r.uniform(0, 500, (n, 2))
    ph = r.integers(0, 4, n)
    f = np.stack([np.sin(pos[:, 0] / 60 + j) + np.cos(pos[:, 1] / 80 + 2 * j) for j in range(6)], 1)
    f = f + r.normal(0, 0.3, f.shape)
    if shuffle:
        f = f[r.permutation(n)]
    _, nn = cKDTree(pos).query(pos, k=CTX_K + 1)
    cf = np.eye(4, dtype=np.float32)[ph[nn[:, 1:]]].mean(1)
    sp = {"syn": list(range(6))}
    spec = {"syn": dict(encs=("intensity", "rank"), modes=(0,), ks=(30,), n_null=5, a3=False)}
    rows, _ = per_image(0, seed, pos, ph, ph % 3, cf, cf[:, :3], ph.astype(np.int16), f.astype(np.float32),
                        (np.array([], int), np.array([], int)), sp, spec)
    d = pd.DataFrame(rows)
    return d[d.metric == "A2"].assign(ex1=lambda x: x.obs - x.n1, ex2=lambda x: x.obs - x.n2)

s_real, s_shuf = synthetic(False), synthetic(True)
print(s_real[["enc", "obs", "n1", "n2", "ex1", "ex2"]].round(3).to_string())
print(s_shuf[["enc", "obs", "n1", "n2", "ex1", "ex2"]].round(3).to_string())
assert (s_real.ex1 < -0.3).all() and (s_real.ex2 < -0.3).all(), "sanity (a) failed"
assert (s_shuf.ex1.abs() < 0.05).all() and (s_shuf.ex2.abs() < 0.05).all(), "sanity (b) failed"
print("sanity checks passed")

## Load data

In [ ]:
report = json.load(open(find_input("prep_report.json")))
# This analysis uses only the cell table and the contact graph (no masks, no clinical data).
print("prep checks:", report["checks"])
assert report["n_cells"] > 1_000_000 and report["checks"]["all_table_images_on_kaggle"]
cells = pd.read_parquet(find_input("cells.parquet"))
contacts = pd.read_parquet(find_input("cell_neighbours.parquet"))
chan = list(report["channel_map"].values())
PROTEIN = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
assert len(PROTEIN) == 36 and set(CYTOF6) <= set(PROTEIN)
print(cells.shape, contacts.shape, f"{N_JOBS} cpus")

## Cohort, broad classes, encodings

In [ ]:
cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
n_per = cells.groupby("image_name").size()
cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
print(f"{len(cells):,} cells, {cells.image_name.nunique()} images, {cells.metabric_id.nunique()} patients")

epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
def broad_class(p):
    if epi_frac[p] > 0.5:
        return "epithelial"
    return "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune"
BROAD = {p: broad_class(p) for p in epi_frac.index}
print(pd.DataFrame({"broad": BROAD, "epi_frac": epi_frac.round(2),
                    "n": cells.cell_phenotype.value_counts()}).sort_values("broad").to_string())

ph_code, ph_names = pd.factorize(cells.cell_phenotype, sort=True)
broad_names = np.array(["epithelial", "immune", "stromal"])
broad_code = pd.Series(cells.cell_phenotype.map(BROAD)).map({b: i for i, b in enumerate(broad_names)}).values
img_code, img_names = pd.factorize(cells.image_name, sort=True)
img_patient = cells.groupby("image_name").metabric_id.first().reindex(img_names).values
starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
xy = cells[["x", "y"]].values.astype(np.float64)

A = np.arcsinh(cells[PROTEIN].values.astype(np.float64) / COFACTOR)
Z = np.empty_like(A, dtype=np.float32)
for a, b in zip(starts[:-1], starts[1:]):
    med = np.median(A[a:b], 0)
    mad = np.median(np.abs(A[a:b] - med), 0)
    Z[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
del A

rng = np.random.default_rng(SEED)
col = {c: i for i, c in enumerate(PROTEIN)}
SPACES = {"cytof6": [col[c] for c in CYTOF6], "full36": list(range(36))}
seen = {tuple(sorted(SPACES["cytof6"]))}
for size, n_sub in SUBSET_SIZES.items():
    got = 0
    while got < n_sub:
        s = tuple(sorted(rng.choice(36, size, replace=False).tolist()))
        if s not in seen:
            seen.add(s)
            SPACES[f"rand{size}_{got:02d}"] = list(s)
            got += 1
MAIN = ("cytof6", "full36")
def size_of(space):
    return 36 if space == "full36" else 6 if space == "cytof6" else int(space[4:].split("_")[0])
SPEC = {s: dict(encs=("intensity", "rank"),
                modes=(0, 1) if s in MAIN else (1,), ks=KS if s in MAIN else (30,),
                n_null=N_NULL if s in MAIN else N_NULL_SUBSET, a3=s in MAIN) for s in SPACES}
json.dump({s: [PROTEIN[i] for i in v] for s, v in SPACES.items()}, open(f"{OUT}/spaces.json", "w"), indent=1)

## Physical context and cellular neighbourhoods

In [ ]:
P_FINE = len(ph_names)
ctx_f = np.zeros((len(cells), P_FINE), np.float32)
ctx_b = np.zeros((len(cells), 3), np.float32)
for a, b in zip(starts[:-1], starts[1:]):
    _, nn = cKDTree(xy[a:b]).query(xy[a:b], k=CTX_K + 1)
    nn = nn[:, 1:] + a
    ctx_f[a:b] = np.eye(P_FINE, dtype=np.float32)[ph_code[nn]].mean(1)
    ctx_b[a:b] = np.eye(3, dtype=np.float32)[broad_code[nn]].mean(1)
sub = rng.choice(len(cells), min(200_000, len(cells)), replace=False)
km = MiniBatchKMeans(N_CN, random_state=SEED, n_init=5, batch_size=8192).fit(ctx_f[sub])
cn = km.predict(ctx_f).astype(np.int16)
print("CN sizes:", np.bincount(cn))

# contact pairs -> local indices per image
loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
    [cells.image_number.values, cells.object_number.values]))
ca = loc.reindex(pd.MultiIndex.from_arrays([contacts.ImageNumber.values, contacts["from"].values])).values
cb = loc.reindex(pd.MultiIndex.from_arrays([contacts.ImageNumber.values, contacts["to"].values])).values
ok = ~(np.isnan(ca) | np.isnan(cb))
ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
pair_img = img_code[ca]
o = np.argsort(pair_img, kind="stable")
ca, cb, pair_img = ca[o], cb[o], pair_img[o]
bnd = np.searchsorted(pair_img, np.arange(len(img_names) + 1))
pairs_by_img = {i: (ca[bnd[i]:bnd[i + 1]] - starts[i], cb[bnd[i]:bnd[i + 1]] - starts[i]) for i in range(len(img_names))}
print(f"{ok.sum():,} contact pairs mapped")

## Run all images

In [ ]:
t0 = time.time()
EMPTY = (np.array([], np.int64), np.array([], np.int64))
res = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(per_image)(i, SEED + i, xy[a:b], ph_code[a:b], broad_code[a:b], ctx_f[a:b], ctx_b[a:b], cn[a:b], Z[a:b],
                       pairs_by_img.get(i, EMPTY), SPACES, SPEC)
    for i, (a, b) in enumerate(zip(starts[:-1], starts[1:])))
print(f"per-image analysis: {(time.time() - t0) / 60:.1f} min")
per_img = pd.DataFrame([r for rows, _ in res for r in rows])
a3 = pd.DataFrame([r for _, rows in res for r in rows])
for d in (per_img, a3):
    d["image"] = img_names[d.img.values]
    d["patient"] = img_patient[d.img.values]
per_img.to_parquet(f"{OUT}/per_image_metrics.parquet", index=False)
a3.to_parquet(f"{OUT}/a3_bands.parquet", index=False)

## Cohort pool (transfer analogue)

In [ ]:
pats = np.array(sorted(set(img_patient)))
rng_p = np.random.default_rng(SEED + 1)
rng_p.shuffle(pats)
half = set(pats[: len(pats) // 2])
cell_pat = cells.metabric_id.values
qi = rng_p.permutation(np.flatnonzero(np.isin(cell_pat, list(half))))[:POOL_Q]
ri = rng_p.permutation(np.flatnonzero(~np.isin(cell_pat, list(half))))[:POOL_R]
ref_by_ph = {p: np.flatnonzero(ph_code[ri] == p) for p in np.unique(ph_code[ri])}

def pool_metrics(P):  # P indexes into ri
    pf = ctx_f[ri][P].mean(1); pb = ctx_b[ri][P].mean(1)
    return dict(se_f=((ctx_f[qi] - pf) ** 2).sum(1), se_b=((ctx_b[qi] - pb) ** 2).sum(1),
                B2=(cn[ri][P] == cn[qi][:, None]).mean(1))

pool_rows = []
for sname in MAIN:
    Zs = Z[:, SPACES[sname]]
    for enc in ("intensity", "rank"):
        V = Zs if enc == "intensity" else rank_vec(Zs)
        jit = rng_p.normal(0, 1e-5, V.shape).astype(np.float32)
        nnm = NearestNeighbors(n_neighbors=30, algorithm="brute", n_jobs=-1).fit(V[ri] + jit[ri])
        P = nnm.kneighbors(V[qi] + jit[qi], return_distance=False)
        out = {"obs": pool_metrics(P)}
        n1 = [pool_metrics(rng_p.integers(0, len(ri), P.shape)) for _ in range(POOL_NULL)]
        n2 = []
        for _ in range(POOL_NULL):
            Q = np.empty_like(P); phP = ph_code[ri][P]
            for p, idx in ref_by_ph.items():
                m = phP == p
                Q[m] = idx[rng_p.integers(0, len(idx), m.sum())]
            n2.append(pool_metrics(Q))
        out["n1"] = {m: np.mean([d[m] for d in n1], 0) for m in out["obs"]}
        out["n2"] = {m: np.mean([d[m] for d in n2], 0) for m in out["obs"]}
        for which, d in out.items():
            pool_rows.append(pd.DataFrame(dict(space=sname, enc=enc, which=which, patient=cell_pat[qi],
                                               se_f=d["se_f"], se_b=d["se_b"], B2=d["B2"],
                                               ss_f=((ctx_f[qi] - ctx_f[qi].mean(0)) ** 2).sum(1),
                                               ss_b=((ctx_b[qi] - ctx_b[qi].mean(0)) ** 2).sum(1))))
        print(sname, enc, "done")
pool = pd.concat(pool_rows)
pool.to_parquet(f"{OUT}/cohort_pool_cells.parquet", index=False)

## Patient bootstrap and summary

In [ ]:
per_img["ex1"] = per_img.obs - per_img.n1
per_img["ex2"] = per_img.obs - per_img.n2
keys = ["space", "enc", "contact_excl", "k", "metric"]
pp = per_img.groupby(keys + ["patient"])[["obs", "n1", "n2", "ex1", "ex2"]].mean().reset_index()
rows = []
for key, g in pp.groupby(keys):
    row = dict(zip(keys, key), n_patients=len(g))
    for c in ["obs", "n1", "n2", "ex1", "ex2"]:
        m, lo, hi = boot_mean(g[c].values)
        row.update({c: m, f"{c}_lo": lo, f"{c}_hi": hi})
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/summary_table.csv", index=False)

# encoding contrast (rank - intensity) on per-patient excess over N2
contrast = []
for (space, cm, k, met), g in pp[pp.space.isin(MAIN)].groupby(["space", "contact_excl", "k", "metric"]):
    w = g.pivot(index="patient", columns="enc", values="ex2")
    for enc in ("rank", "rank_avg"):
        if enc in w:
            m, lo, hi = boot_mean((w[enc] - w["intensity"]).values)
            contrast.append(dict(space=space, contact_excl=cm, k=k, metric=met, contrast=f"{enc}-intensity",
                                 diff=m, lo=lo, hi=hi))
contrast = pd.DataFrame(contrast)
contrast.to_csv(f"{OUT}/encoding_contrast.csv", index=False)

# cohort pool: R^2 from pooled SE over resampled patients
pool_sum = []
for (space, enc), g in pool.groupby(["space", "enc"]):
    per_p = g.groupby(["which", "patient"])[["se_f", "ss_f", "se_b", "ss_b", "B2"]].agg(
        {"se_f": "sum", "ss_f": "sum", "se_b": "sum", "ss_b": "sum", "B2": "mean"}).reset_index()
    wide = {w: per_p[per_p.which == w].set_index("patient") for w in ("obs", "n1", "n2")}
    pats_q = wide["obs"].index.values
    r = np.random.default_rng(SEED)
    boots = r.integers(0, len(pats_q), (N_BOOT, len(pats_q)))
    for met, (se, ss) in {"B1f": ("se_f", "ss_f"), "B1b": ("se_b", "ss_b")}.items():
        vals = {w: 1 - wide[w][se].values[boots].sum(1) / wide[w][ss].values[boots].sum(1) for w in wide}
        full = {w: 1 - wide[w][se].sum() / wide[w][ss].sum() for w in wide}
        for lab, a, bb in [("obs", "obs", None), ("ex1", "obs", "n1"), ("ex2", "obs", "n2")]:
            v = vals[a] - (vals[bb] if bb else 0)
            f = full[a] - (full[bb] if bb else 0)
            pool_sum.append(dict(space=space, enc=enc, metric=met, stat=lab, value=f,
                                 lo=np.percentile(v, 2.5), hi=np.percentile(v, 97.5)))
    for lab, a, bb in [("obs", "obs", None), ("ex1", "obs", "n1"), ("ex2", "obs", "n2")]:
        v = wide[a]["B2"].values - (wide[bb]["B2"].values if bb else 0)
        m, lo, hi = boot_mean(v)
        pool_sum.append(dict(space=space, enc=enc, metric="B2", stat=lab, value=m, lo=lo, hi=hi))
pool_sum = pd.DataFrame(pool_sum)
pool_sum.to_csv(f"{OUT}/cohort_pool_summary.csv", index=False)

## Decision rule (evaluated as pre-registered)

In [ ]:
def get(space, enc, metric, stat, cm=1, k=30):
    r = summary[(summary.space == space) & (summary.enc == enc) & (summary.metric == metric)
                & (summary.contact_excl == cm) & (summary.k == k)].iloc[0]
    return float(r[stat]), float(r[f"{stat}_lo"]), float(r[f"{stat}_hi"])

decision = {}
for space in MAIN:
    for enc in ("intensity", "rank"):
        a2_1, a2_2, b1_2 = get(space, enc, "A2", "ex1"), get(space, enc, "A2", "ex2"), get(space, enc, "B1f", "ex2")
        decision[f"{space}/{enc}"] = dict(
            A2_ex1=a2_1, A2_ex2=a2_2, B1f_ex2=b1_2,
            closer_than_chance=a2_1[2] < 0, closer_beyond_type=a2_2[2] < 0,
            context_beyond_type=b1_2[1] > 0)
ctx = {s: any(decision[f"{s}/{e}"]["context_beyond_type"] for e in ("intensity", "rank")) for s in MAIN}
if not ctx["cytof6"] and not ctx["full36"]:
    verdict = "NO physical context beyond cell type in either panel -> drop the CyTOF virtual-space transfer (Step 3)"
elif ctx["full36"] and not ctx["cytof6"]:
    verdict = "Context beyond cell type only with the full 36-marker panel -> panel-limited; CyTOF 31-marker panel untestable here"
else:
    verdict = "Profile neighbours carry physical context beyond cell type with the CyTOF-matched markers"
enc_answer = contrast[(contrast.k == 30) & (contrast.contact_excl == 1) & contrast.metric.isin(["B1f", "A2"])]
out = dict(verdict=verdict, decision=decision, encoding_contrast=enc_answer.to_dict("records"),
           n_cells=int(len(cells)), n_images=int(len(img_names)), n_patients=int(len(set(img_patient))),
           cohort_pool=pool_sum.to_dict("records"))
json.dump(out, open(f"{OUT}/summary.json", "w"), indent=1, default=float)
print(verdict)
print(pd.DataFrame(decision).T.to_string())
print(enc_answer.round(4).to_string())

## Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
mids = ["0-15", "15-30", "30-60", "60-120", "120-240", ">240"]
a3p = a3.groupby(["space", "enc", "band", "patient"]).rel.mean().reset_index()
for ax, space in zip(axes, MAIN):
    for enc, c in zip(("intensity", "rank"), ("C0", "C1")):
        g = a3p[(a3p.space == space) & (a3p.enc == enc)]
        st = [boot_mean(g[g.band == b].rel.values) for b in range(6)]
        m = [s[0] for s in st]; lo = [s[1] for s in st]; hi = [s[2] for s in st]
        ax.plot(range(6), m, "o-", color=c, label=enc)
        ax.fill_between(range(6), lo, hi, color=c, alpha=0.2)
    ax.axhline(1, color="k", lw=0.8, ls="--")
    ax.set_xticks(range(6)); ax.set_xticklabels(mids)
    ax.set_xlabel("physical distance (µm)"); ax.set_title(space)
axes[0].set_ylabel("profile distance / image mean")
axes[0].legend()
fig.tight_layout(); fig.savefig(f"{OUT}/fig_distance_decay.png", dpi=150)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, met in zip(axes, ("A2", "B1f")):
    for j, enc in enumerate(("intensity", "rank")):
        s = summary[(summary.metric == met) & (summary.enc == enc) & (summary.k == 30) & (summary.contact_excl == 1)]
        rnd = s[s.space.str.startswith("rand6_")]
        ax.scatter(np.full(len(rnd), j) + np.random.default_rng(0).uniform(-0.12, 0.12, len(rnd)),
                   rnd.ex2, s=14, color="grey", alpha=0.7, label="random 6" if j == 0 else None)
        for space, mk, c in (("cytof6", "D", "C3"), ("full36", "s", "C0")):
            r = s[s.space == space].iloc[0]
            ax.errorbar(j + (0.25 if space == "full36" else -0.25), r.ex2,
                        yerr=[[r.ex2 - r.ex2_lo], [r.ex2_hi - r.ex2]], fmt=mk, color=c,
                        label=space if j == 0 else None)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["intensity", "rank"])
    ax.set_title(f"{met}: excess over type-matched null (k=30, contacts excl.)")
axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/fig_panel_size.png", dpi=150)

fig, ax = plt.subplots(figsize=(9, 5))
s = summary[summary.space.isin(MAIN) & (summary.k == 30) & (summary.contact_excl == 1)
            & summary.metric.isin(["A1", "A2", "B1f", "B1b", "B2"])].reset_index(drop=True)
lab = s.metric + " | " + s.space + " | " + s.enc
ax.errorbar(s.ex2, range(len(s)), xerr=[s.ex2 - s.ex2_lo, s.ex2_hi - s.ex2], fmt="o", label="obs − N2 (beyond type)")
ax.errorbar(s.ex1, np.arange(len(s)) + 0.3, xerr=[s.ex1 - s.ex1_lo, s.ex1_hi - s.ex1], fmt="s", ms=3, label="obs − N1")
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(range(len(s))); ax.set_yticklabels(lab, fontsize=7)
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig_excess_forest.png", dpi=150)

## Panel-size curve: does ranking lose information as the panel shrinks?
For each panel size, per-patient values are averaged over that size's random marker sets (size 36 = the full panel).
The rank − intensity difference is bootstrapped over patients.
**No harm** (fixed before the run) means the CI of (rank − intensity) lies above −0.01, for context R² (B1f) and type purity (T1).

In [ ]:
cur = pp[(pp.k == 30) & (pp.contact_excl == 1) & (pp.space.str.startswith("rand") | (pp.space == "full36"))].copy()
cur["size"] = cur.space.map(size_of)
curve_rows = []
for (size, met), g in cur.groupby(["size", "metric"]):
    per_pat = g.groupby(["patient", "enc"])[["obs", "ex1", "ex2"]].mean().unstack("enc")
    for stat in ("obs", "ex2"):
        row = dict(size=size, metric=met, stat=stat)
        for enc in ("intensity", "rank"):
            row[enc], row[f"{enc}_lo"], row[f"{enc}_hi"] = boot_mean(per_pat[(stat, enc)].values)
        row["diff"], row["diff_lo"], row["diff_hi"] = boot_mean((per_pat[(stat, "rank")] - per_pat[(stat, "intensity")]).values)
        curve_rows.append(row)
curve = pd.DataFrame(curve_rows)
curve["no_harm"] = curve.diff_lo > -0.01
curve.to_csv(f"{OUT}/panel_size_curve.csv", index=False)
print(curve[curve.metric.isin(["A2", "B1f", "T1", "T1b"])].round(4).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panels = [("B1f", "obs", "context R² (fine)"), ("T1", "obs", "same-phenotype neighbours"),
          ("A2", "obs", "log2 distance ratio (lower = closer)")]
for j, (met, stat, lab) in enumerate(panels):
    g = curve[(curve.metric == met) & (curve.stat == stat)].sort_values("size")
    ax = axes[0, j]
    for enc, c in (("intensity", "C0"), ("rank", "C1")):
        ax.plot(g["size"], g[enc], "o-", color=c, label=enc)
        ax.fill_between(g["size"], g[f"{enc}_lo"], g[f"{enc}_hi"], color=c, alpha=0.2)
    ax.set_title(lab); ax.set_xlabel("markers in panel"); ax.legend()
    ax = axes[1, j]
    ax.plot(g["size"], g["diff"], "o-", color="k")
    ax.fill_between(g["size"], g["diff_lo"], g["diff_hi"], color="k", alpha=0.2)
    ax.axhline(0, color="grey", lw=0.8); ax.axhline(-0.01, color="r", lw=0.8, ls="--")
    ax.axvline(31, color="grey", lw=0.8, ls=":")
    ax.set_title(f"rank − intensity: {lab}"); ax.set_xlabel("markers in panel (dotted = CyTOF panel size)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_panel_size_curve.png", dpi=150)

print("done")